# 03 - Model Experiments

Loads and compares the metrics produced by `src.pipeline.training_pipeline` for:

- **Experiment A**: price/technical features only (`run-name=experiment_a_price_only`)
- **Experiment B**: price/technical features + sentiment features (`run-name=experiment_b_price_sentiment`)

Reproduce these runs from the `ml/` directory with:
```
python -m src.pipeline.training_pipeline --run-name experiment_a_price_only
python -m src.pipeline.training_pipeline --run-name experiment_b_price_sentiment --use-sentiment
```

**Important caveat**: the sample dataset used here is *synthetically generated* (`scripts/generate_sample_data.py`) with sentiment intentionally correlated to future returns, purely so the pipeline can be exercised end-to-end without external data access. Any "sentiment helps" or "sentiment doesn't help" conclusion drawn from this notebook reflects the synthetic data-generating process, not a genuine market effect. Re-run this comparison on the real datasets referenced in the README once they are available to draw real conclusions.

In [ ]:
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd

from src.config import ML_ROOT

reports_root = ML_ROOT / "reports"

with open(reports_root / "experiment_a_price_only" / "metrics.json") as f:
    metrics_a = json.load(f)
with open(reports_root / "experiment_b_price_sentiment" / "metrics.json") as f:
    metrics_b = json.load(f)

In [ ]:
rows = []
for exp_name, metrics in [("A (price only)", metrics_a), ("B (price + sentiment)", metrics_b)]:
    for model_name, m in metrics.items():
        rows.append({"experiment": exp_name, "model": model_name, **m})

comparison_df = pd.DataFrame(rows)
comparison_df[["experiment", "model", "mae", "rmse", "mape", "r2", "directional_accuracy", "f1_up", "f1_down"]]

## Deep model (LSTM) comparison: does sentiment help?

In [ ]:
lstm_a = metrics_a["lstm"]
lstm_b = metrics_b["lstm"]

print("Metric           | A (price only) | B (price + sentiment) | Delta")
for key in ["mae", "rmse", "directional_accuracy", "f1_up", "f1_down"]:
    delta = lstm_b[key] - lstm_a[key]
    print(f"{key:<16} | {lstm_a[key]:>14.6f} | {lstm_b[key]:>21.6f} | {delta:+.6f}")

## View saved training/validation loss curves and prediction plots

In [ ]:
from IPython.display import Image, display

for run in ["experiment_a_price_only", "experiment_b_price_sentiment"]:
    print(run)
    display(Image(filename=str(reports_root / run / "loss_curve.png")))
    display(Image(filename=str(reports_root / run / "actual_vs_predicted.png")))